# Homework 3 — Computational Component

**CHMENG 250: Transport Processes**

## Steady Couette–Poiseuille Flow by Finite Differences

### Instructions

Work through this notebook **sequentially, one cell at a time**.

* If you are using the recommended `coursework-env` environment, select **Python (coursework-env)** as the notebook kernel.
* Run the first code cell, which imports the required Python packages, `numpy` and `matplotlib`. You do not need to modify this cell.
* In subsequent code cells, follow the instructions marked by `TODO` comments. This may involve replacing `...`, filling in the requested code, or completing and uncommenting a provided line of code.
* After completing all `TODO` items in a cell, if the cell contains a `raise NotImplementedError(...)` line, **comment it out before running the cell**.
* Proceed to the next cell only after the current cell has been completed and **runs without errors**, since later cells may depend on variables, functions, or results defined earlier in the notebook.
* Questions requiring a written response are shown in <span style="color:red">**red**</span>. Enter your response after **Answer:** in the space provided. Questions will appear in the following format:

<span style="color:red">

**Question:** Example question requiring a written response.

</span>

**Answer:**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Compatibility across NumPy versions: use np.trapezoid when available,
# otherwise np.trapz
trapz = np.trapezoid if hasattr(np, 'trapezoid') else np.trapz

---
## Part 1 — Problem Setup and Finite-Difference Solution

### The physical problem

Consider steady, fully developed, incompressible flow of a Newtonian fluid between two infinite parallel plates separated by a gap $H$. The bottom plate at $y=0$ is stationary, while the top plate at $y=H$ moves with velocity $U$ in the $x$-direction. A constant streamwise pressure gradient

$$G = \frac{\mathrm{d}p}{\mathrm{d}x}$$

provides an additional pressure-driven contribution to the flow. With this sign convention, $G<0$ corresponds to pressure decreasing in the $+x$-direction.

The momentum equation reduces to the ODE

$$\mu \frac{\mathrm{d}^2 u}{\mathrm{d}y^2} = G, \qquad u(0)=0,\quad u(H)=U.$$

**Analytical solution:**

$$u(y) = \frac{U y}{H} + \frac{G}{2\mu}\,y(y-H).$$

The first term is the Couette (plate-driven) contribution, while the second is the Poiseuille (pressure-driven) contribution.

### 1(a) Parameters and analytical solution

Define the physical parameters below. With $G<0$ (favorable pressure gradient), both the plate motion and the pressure gradient drive flow in the $+x$-direction.

In [ ]:
# Physical parameters
H = 1.0      # plate separation
U = 1.0      # top-plate velocity
mu = 0.1     # dynamic viscosity
G = -1.0     # pressure gradient dp/dx (< 0 is favorable)

# Uniform grid in the y-direction
N = 50       # number of intervals (N + 1 grid points)
y = np.linspace(0, H, N + 1)
dy = y[1] - y[0]

# TODO: Evaluate the analytical velocity profile on the grid
u_exact = ...
raise NotImplementedError("Compute u_exact")

### 1(b) Assemble the finite-difference system

Discretize $\mathrm{d}^2u/\mathrm{d}y^2$ at the interior grid points using the central-difference stencil:

$$\frac{u_{j-1}-2u_j+u_{j+1}}{\Delta y^2}=\frac{G}{\mu}, \qquad j=1,\ldots,N-1.$$

Applying this stencil at each interior grid point gives a tridiagonal linear system $\boldsymbol{A}\boldsymbol{u}=\boldsymbol{b}$ for the $N-1$ interior unknowns $u_1,\ldots,u_{N-1}$. The boundary values $u_0=0$ and $u_N=U$ enter the right-hand side $\boldsymbol{b}$.

In [ ]:
n = N - 1  # number of interior unknowns
coeff = 1.0 / dy**2

# Each interior finite-difference equation couples u_{j-1}, u_j, and u_{j+1}.
# TODO: Build the tridiagonal matrix A (n x n)
# Diagonal = -2/dy^2, off-diagonals = 1/dy^2
A = ...

# Start with the forcing G/mu at every interior grid point.
# The first and last equations also contain known boundary values,
# whose contributions must be moved to the right-hand side.
# TODO: Build the RHS vector b and apply the boundary-condition corrections
b = ...

raise NotImplementedError("Build the finite-difference system")

### 1(c) Solve and compare with the analytical solution

Solve the linear system, reconstruct the full numerical velocity profile including the boundary values, and plot it against the analytical solution.

In [ ]:
# TODO: Solve the linear system for the interior velocity values
u_interior = ...

# Reconstruct the full velocity profile by inserting the prescribed boundary values
u_num = np.concatenate([[0.0], u_interior, [U]])

raise NotImplementedError("Solve the linear system")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 6))

ax.plot(u_exact, y, 'k-', lw=2, label='Analytical')
ax.plot(u_num, y, 'ro', ms=4, markevery=2, label='Finite difference')

ax.set_xlabel('$u(y)$')
ax.set_ylabel('$y$')
ax.set_title('Couette–Poiseuille velocity profile')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Max |u_num - u_exact|: {np.max(np.abs(u_num - u_exact)):.2e}")

<span style="color:red">

**Question:** The error is at machine precision ($\sim 10^{-14}$). Why? *Hint:* the analytical solution is a quadratic polynomial. What is the truncation error of the central-difference approximation to $\mathrm{d}^2u/\mathrm{d}y^2$, and why does it vanish for this solution?

</span>

**Answer:**

---
## Part 2 — 2D Velocity Field Visualization

The flow is fully developed, so the velocity field is independent of $x$:

$$\boldsymbol{u}(x,y)=(u(y),0).$$

Although the velocity profile depends only on $y$, we can represent it over a two-dimensional $(x,y)$ domain to visualize the flow field.

### 2(a) Quiver plot

Create a two-dimensional grid and replicate the one-dimensional velocity profile at each $x$-station. Use `quiver` to visualize the velocity vectors. Since $v=0$ everywhere, all arrows are horizontal, with their lengths indicating the local streamwise velocity $u(y)$.

In [ ]:
# TODO: Build the 2D velocity field
Nx = 11
x_2d = np.linspace(0, 4 * H, Nx)
X2, Y2 = np.meshgrid(x_2d, y)

# Repeat the 1D streamwise profile at every x-station so U2 has the same
# shape as X2 and Y2
U2 = ...

# The transverse velocity is zero everywhere for fully developed flow
V2 = ...

raise NotImplementedError("Build 2D velocity field")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Velocity magnitude used for the background color map
speed2 = np.hypot(U2, V2)

# Plot every second grid point in the y-direction to reduce arrow crowding
s = 2

im = ax.pcolormesh(
    X2, Y2, speed2,
    cmap='viridis',
    shading='auto',
    alpha=0.6
)

ax.quiver(
    X2[::s, :], Y2[::s, :],
    U2[::s, :], V2[::s, :],
    color='k'
)

fig.colorbar(im, ax=ax, label=r'$|\boldsymbol{u}|$')

ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_title('2D velocity field (Couette–Poiseuille)')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

### 2(b) Streamlines

For fully developed flow with $v=0$, the streamlines are horizontal lines $y=\text{const}$. The coloring indicates the local speed.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

strm = ax.streamplot(
    x_2d, y, U2, V2,
    color=speed2,
    cmap='viridis',
    density=2
)

fig.colorbar(strm.lines, ax=ax, label=r'$|\boldsymbol{u}|$')

ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_title('Streamlines')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

<span style="color:red">

**Question:** At what value of $y$ is the velocity maximum? Does this location coincide with either wall? Why or why not?

</span>

**Answer:**

---
## Part 3 — Stress, Vorticity, and Dissipation

All three quantities are determined by the velocity gradient $\mathrm{d}u/\mathrm{d}y$.

### 3(a) Compute $\mathrm{d}u/\mathrm{d}y$ numerically

Use **central differences** at the interior grid points:

$$\left.\frac{\mathrm{d}u}{\mathrm{d}y}\right|_j \approx \frac{u_{j+1}-u_{j-1}}{2\Delta y}, \qquad j=1,\ldots,N-1.$$

At the walls, use **second-order one-sided differences**:

$$\left.\frac{\mathrm{d}u}{\mathrm{d}y}\right|_0 \approx \frac{-3u_0+4u_1-u_2}{2\Delta y}, \qquad \left.\frac{\mathrm{d}u}{\mathrm{d}y}\right|_N \approx \frac{3u_N-4u_{N-1}+u_{N-2}}{2\Delta y}.$$

Both the interior and wall approximations are second-order accurate.

In [ ]:
# TODO: Compute du/dy at all N + 1 grid points
dudy_num = np.zeros(N + 1)

# Interior points: use central differences
# dudy_num[1:-1] = ...

# Wall points: use second-order one-sided differences
# dudy_num[0] = ...
# dudy_num[-1] = ...

raise NotImplementedError("Compute numerical velocity gradient")

### 3(b) Shear stress and vorticity profiles

The shear-stress component is $\tau_{xy}(y)=\mu\,\mathrm{d}u/\mathrm{d}y$. For this two-dimensional flow, the only nonzero component of the vorticity $\boldsymbol{\omega}=\nabla\times\boldsymbol{u}$ is the $z$-component,

$$\omega_z(y)=-\frac{\mathrm{d}u}{\mathrm{d}y}.$$

Analytically,

$$\frac{\mathrm{d}u}{\mathrm{d}y}=\frac{U}{H}+\frac{G}{\mu}\left(y-\frac{H}{2}\right),$$

so both $\tau_{xy}$ and $\omega_z$ vary linearly with $y$.

In [ ]:
# TODO: Evaluate the analytical velocity gradient on the grid
dudy_exact = ...

raise NotImplementedError("Compute analytical velocity gradient")

tau_exact = mu * dudy_exact
tau_num = mu * dudy_num

omega_z_exact = -dudy_exact
omega_z_num = -dudy_num

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Shear stress
axes[0].plot(tau_exact, y, 'k-', lw=2, label='Analytical')
axes[0].plot(tau_num, y, 'ro', ms=4, markevery=2, label='Numerical')
axes[0].axvline(0, color='gray', ls=':', lw=0.8)
axes[0].set_xlabel(r'$\tau_{xy}(y)$')
axes[0].set_ylabel(r'$y$')
axes[0].set_title('Shear stress')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Vorticity
axes[1].plot(omega_z_exact, y, 'k-', lw=2, label='Analytical')
axes[1].plot(omega_z_num, y, 'ro', ms=4, markevery=2, label='Numerical')
axes[1].axvline(0, color='gray', ls=':', lw=0.8)
axes[1].set_xlabel(r'$\omega_z(y)$')
axes[1].set_ylabel(r'$y$')
axes[1].set_title('Vorticity')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(
    f"Shear-stress component at walls: "
    f"tau_xy(0) = {tau_num[0]:.4f}, tau_xy(H) = {tau_num[-1]:.4f}"
)

<span style="color:red">

**Question:** For the parameters used here, the shear-stress component $\tau_{xy}=\mu\,\mathrm{d}u/\mathrm{d}y$ has opposite signs at the two walls. The outward unit normals of the fluid are $-\boldsymbol{e}_y$ at $y=0$ and $+\boldsymbol{e}_y$ at $y=H$. Using $\boldsymbol{t}=\boldsymbol{\sigma}\cdot\boldsymbol{n}$, what are the directions of the tangential tractions exerted on the fluid at the two walls? Why do opposite signs of $\tau_{xy}$ not necessarily imply opposite traction directions?

</span>

**Answer:**

### 3(c) Viscous dissipation

For this flow, the viscous dissipation rate per unit volume is

$$\Phi(y)=\mu\left(\frac{\mathrm{d}u}{\mathrm{d}y}\right)^2.$$

This quantity indicates where mechanical energy is irreversibly converted into internal energy by viscous stresses.

In [ ]:
# TODO: Compute the analytical and numerical viscous dissipation rates
Phi_exact = ...
Phi_num = ...

raise NotImplementedError("Compute viscous dissipation")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

ax.plot(Phi_exact, y, 'k-', lw=2, label='Analytical')
ax.plot(Phi_num, y, 'ro', ms=4, markevery=2, label='Numerical')

ax.set_xlabel(r'$\Phi(y)=\mu\left(\mathrm{d}u/\mathrm{d}y\right)^2$')
ax.set_ylabel(r'$y$')
ax.set_title('Viscous dissipation rate')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<span style="color:red">

**Question:** Where is the viscous dissipation highest: near the walls or at the location of maximum velocity? Why does this make sense in terms of the velocity gradient?

</span>

**Answer:**

### 3(d) Volume flow rate

The volume flow rate per unit width is $Q=\int_0^H u(y)\,\mathrm{d}y$. Compute it numerically using the trapezoidal rule and compare it with the analytical result:

$$Q_{\mathrm{exact}}=\frac{UH}{2}-\frac{GH^3}{12\mu}.$$

In [ ]:
# TODO: Compute the volume flow rate numerically and analytically
# Use trapz(u_num, y), where trapz is the compatibility alias defined above
Q_trap = ...
Q_exact = ...

raise NotImplementedError("Compute volume flow rate")

In [ ]:
print(f"Q (trapezoidal): {Q_trap:.6f}")
print(f"Q (analytical):   {Q_exact:.6f}")
print(f"Relative error:   {abs(Q_trap - Q_exact) / abs(Q_exact):.2e}")

---
## Part 4 — Convergence Study

### Why we need a new test case

On a uniform grid, the central-difference approximation to $\mathrm{d}^2u/\mathrm{d}y^2$ is exact for polynomials of degree 3 or lower. Its leading truncation error is proportional to $\Delta y^2\,\mathrm{d}^4u/\mathrm{d}y^4$, which vanishes for cubic and lower-degree polynomials. Since the Couette–Poiseuille solution is quadratic, the finite-difference solution differs from the analytical solution only by roundoff error, making it unsuitable for demonstrating convergence under grid refinement.

To obtain a nonzero discretization error, add a **sinusoidal body-force density** in the $-x$-direction:

$$f_x(y)=-A\sin\!\left(\frac{\pi y}{H}\right), \qquad A>0.$$

The steady $x$-momentum balance is

$$0=-G+\mu\frac{\mathrm{d}^2u}{\mathrm{d}y^2}+f_x.$$

Substituting the body force and rearranging gives

$$\mu\frac{\mathrm{d}^2u}{\mathrm{d}y^2}=G+A\sin\!\left(\frac{\pi y}{H}\right).$$

The corresponding analytical solution is

$$u(y)=\left(\frac{U}{H}-\frac{GH}{2\mu}\right)y+\frac{G}{2\mu}y^2-\frac{AH^2}{\mu\pi^2}\sin\!\left(\frac{\pi y}{H}\right).$$

### 4(a) Solve with body force

Modify your finite-difference solver for the body-force case. The tridiagonal matrix has the same structure as before because the discretization of $\mathrm{d}^2u/\mathrm{d}y^2$ is unchanged, while the right-hand side now contains the spatially varying forcing term from the ODE above.

In [ ]:
A_amp = 1.0  # amplitude A of the body-force density

def solve_couette_body(Ni):
    """Solve the Couette–Poiseuille problem with body force on Ni intervals."""
    yi = np.linspace(0, H, Ni + 1)
    dyi = yi[1] - yi[0]
    ni = Ni - 1
    ci = 1.0 / dyi**2

    # TODO: Build the tridiagonal matrix using the same stencil as in Part 1
    Ai = ...

    # TODO: Evaluate the spatially varying RHS at the interior grid points,
    # then apply the boundary-condition corrections
    bi = ...

    # TODO: Solve for the interior velocity values
    u_int = ...

    # Reconstruct the full profile using the prescribed boundary values
    u_n = np.concatenate([[0.0], u_int, [U]])

    # Evaluate the analytical solution for the body-force case
    u_ex = ((U / H - G * H / (2 * mu)) * yi
            + (G / (2 * mu)) * yi**2
            - (A_amp * H**2 / (mu * np.pi**2)) * np.sin(np.pi * yi / H))

    return yi, u_n, u_ex

raise NotImplementedError("Implement solve_couette_body")

In [ ]:
# Quick check: solve on N = 50 intervals and compare with the analytical solution
y_check, u_check, u_ex_check = solve_couette_body(50)

fig, ax = plt.subplots(figsize=(5, 6))

ax.plot(u_ex_check, y_check, 'k-', lw=2, label='Analytical')
ax.plot(u_check, y_check, 'ro', ms=4, markevery=2, label='Finite difference')

ax.set_xlabel(r'$u(y)$')
ax.set_ylabel(r'$y$')
ax.set_title('Couette–Poiseuille flow with body force')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4(b) Convergence plot

Solve the body-force problem for a range of grid sizes and compute the discrete $L_2$/RMS error in $u(y)$:

$$\varepsilon=\sqrt{\frac{1}{N+1}\sum_{j=0}^{N}\left(u_j^{\mathrm{num}}-u_j^{\mathrm{exact}}\right)^2}.$$

Plot $\varepsilon$ versus $\Delta y=H/N$ on a log-log scale. Fit a straight line to the data and use its slope to verify the expected second-order convergence.

In [ ]:
# Grid sizes for the convergence study
Ns = [10, 20, 40, 80, 160, 320]
errors = []

for Ni in Ns:
    yi, u_n, u_ex = solve_couette_body(Ni)

    # TODO: Compute the discrete L2/RMS error using the definition above
    err = ...
    errors.append(err)

# Corresponding grid spacings
dys = np.array([H / Ni for Ni in Ns])
errors = np.array(errors)

raise NotImplementedError("Complete convergence study")

In [ ]:
# Fit the slope of log10(error) versus log10(grid spacing)
coeffs = np.polyfit(np.log10(dys), np.log10(errors), 1)
slope = coeffs[0]

fig, ax = plt.subplots(figsize=(6, 5))

ax.loglog(
    dys, errors, 'ko-',
    label=f'Finite-difference error (slope = {slope:.2f})'
)

# Second-order reference line anchored at the coarsest-grid error
ax.loglog(
    dys,
    errors[0] * (dys / dys[0])**2,
    'r--',
    label=r'$\propto \Delta y^2$ (reference)'
)

ax.set_xlabel(r'$\Delta y$')
ax.set_ylabel(r'$\varepsilon$ (discrete $L_2$/RMS error)')
ax.set_title('Convergence of the finite-difference solution')
ax.legend()
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Fitted slope: {slope:.3f} (expected: 2.0)")

### 4(c) Discussion questions

<span style="color:red">

**Question 1:** The Couette–Poiseuille solution without the body force gave roundoff-level error, whereas the body-force case does not. Explain this in terms of the truncation error $\propto \Delta y^2\,\mathrm{d}^4u/\mathrm{d}y^4$.

</span>

**Answer:**

<span style="color:red">

**Question 2:** If we used a **fourth-order** compact stencil for $\mathrm{d}^2u/\mathrm{d}y^2$, what convergence rate would you expect? What practical complication arises when constructing a boundary treatment that preserves this order of accuracy?

</span>

**Answer:**

<span style="color:red">

**Question 3:** The composite trapezoidal rule is second-order accurate for a smooth, nonperiodic integrand. For the flow-rate calculation in Part 3, what convergence rate do you expect as $\Delta y$ is refined? Would adding the sinusoidal term change the nominal order of accuracy on $0\le y\le H$?

</span>

**Answer:**